# 🥚 Egg Yolk Color Prediction - Model Training & Evaluation Notebook

สมุดบันทึกประเมินและเปรียบเทียบประสิทธิภาพของโมเดล Machine Learning 5 ตัว
ด้วยวิธี **Stratified 5-Fold Cross-Validation** (รักษาสัดส่วนคลาส 80/20)

คำนวณและแสดงผลตัววัดผลทางสถิติหลักของโมเดล Regression ได้แก่ **Train R², Test R², Train MAE, Test MAE, Train RMSE, และ Test RMSE**

In [1]:
# 1. Import Libraries
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
# 2. Load Features & Stratified 5-Fold Cross-Validation Evaluation (R2, MAE, RMSE)
features_csv = 'data/features.csv'
df = pd.read_csv(features_csv)
print(f"Loaded features dataset: {len(df)} samples across 12 classes.\n")

feature_cols = ['r', 'g', 'b', 'l', 'a', 'b_lab']
X = df[feature_cols].values
y = df['fan_score'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'SVR (RBF Kernel)': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', SVR(kernel='rbf', C=10.0, epsilon=0.1))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge(alpha=1.0))
    ])
}

comparison_results = []

for name, pipeline in models.items():
    tr_r2, te_r2 = [], []
    tr_mae, te_mae = [], []
    tr_rmse, te_rmse = [], []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[val_idx], y[val_idx]

        pipeline.fit(X_tr, y_tr)

        # Predict Train
        p_tr = pipeline.predict(X_tr)
        tr_r2.append(r2_score(y_tr, p_tr))
        tr_mae.append(mean_absolute_error(y_tr, p_tr))
        tr_rmse.append(np.sqrt(mean_squared_error(y_tr, p_tr)))

        # Predict Test (Validation)
        p_te = pipeline.predict(X_te)
        te_r2.append(r2_score(y_te, p_te))
        te_mae.append(mean_absolute_error(y_te, p_te))
        te_rmse.append(np.sqrt(mean_squared_error(y_te, p_te)))

    comparison_results.append({
        'Model': name,
        'Train R2': round(np.mean(tr_r2), 4),
        'Test R2': round(np.mean(te_r2), 4),
        'Train MAE': round(np.mean(tr_mae), 4),
        'Test MAE': round(np.mean(te_mae), 4),
        'Train RMSE': round(np.mean(tr_rmse), 4),
        'Test RMSE': round(np.mean(te_rmse), 4)
    })

res_df = pd.DataFrame(comparison_results).sort_values(by='Test R2', ascending=False)

print("=" * 90)
print("MODEL TRAIN vs TEST COMPARISON RESULTS (Stratified 5-Fold Cross-Validation)")
print("=" * 90)
print(res_df.to_string(index=False))
print("=" * 90)

best_model_name = res_df.iloc[0]['Model']
print(f"\nBest Performing Model: {best_model_name} (Test R^2 = {res_df.iloc[0]['Test R2']:.4f})")

Loaded features dataset: 647 samples across 12 classes.

MODEL TRAIN vs TEST COMPARISON RESULTS (Stratified 5-Fold Cross-Validation)
            Model  Train R2  Test R2  Train MAE  Test MAE  Train RMSE  Test RMSE
 SVR (RBF Kernel)    0.9173   0.9004     0.6233    0.6967      0.8571     0.9393
Gradient Boosting    0.9612   0.8900     0.4474    0.7206      0.5870     0.9875
    Random Forest    0.9847   0.8853     0.2665    0.7326      0.3693     1.0059
Linear Regression    0.8494   0.8448     0.8916    0.8978      1.1570     1.1729
 Ridge Regression    0.8276   0.8246     0.9788    0.9829      1.2381     1.2477

Best Performing Model: SVR (RBF Kernel) (Test R^2 = 0.9004)
